# Data Generation Notebook (Optional)
## How mda_corpus.csv Was Built from SEC EDGAR

### Purpose
This is a companion notebook to 01_SP500_Sentiment_Analysis.ipynb. It documents the full data collection pipeline used to build the MD&A corpus from scratch. You do not need to run this notebook to reproduce the analysis - the pre-built mda_corpus.csv is provided in the repository.

This notebook exists for three reasons:
1. Full transparency about data provenance and collection methodology

2. Reproducibility for anyone who wants to verify or extend the corpus

3. Academic credibility - the data comes from the SEC primary source, not a third-party dataset

### Runtime Warning
- Phase 1 (filing index): approximately 5-10 minutes

- Phase 2 (MD&A extraction): approximately 30-60 minutes

- Both phases cache their output to CSV files and skip on re-runs

### What Gets Generated
- data/filing_index.csv: metadata for all 10-K filings found on EDGAR

- data/mda_corpus.csv: extracted and cleaned MD&A text for each filing

# Setup

In [ ]:
import sys
!{sys.executable} -m pip install -q requests beautifulsoup4 pandas tqdm

In [ ]:
import re
import time
import requests
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from tqdm import tqdm

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

START_YEAR = 1995
END_YEAR   = 2024

# SEC requires a descriptive User-Agent with contact details
EDGAR_HEADERS = {
    "User-Agent": "MSc Research Project research@university.ac.uk",
    "Accept-Encoding": "gzip, deflate",
}

print("Setup complete")

# Phase 1: Build Filing Index from SEC EDGAR

Query the EDGAR submissions API for each company and collect 10-K filing metadata.
The API endpoint is: https://data.sec.gov/submissions/CIK{cik}.json

Each company has a unique CIK (Central Index Key). We filter for annual 10-K filings
only and store the index page URL for downstream document retrieval.

In [ ]:
# Company universe - 76 S&P 500 companies across all 11 GICS sectors
SP500_COMPANIES = {
    "AAPL":  {"cik": "0000320193", "name": "Apple Inc.",               "sector": "Technology"},
    "MSFT":  {"cik": "0000789019", "name": "Microsoft Corp.",          "sector": "Technology"},
    "GOOGL": {"cik": "0001652044", "name": "Alphabet Inc.",            "sector": "Technology"},
    "AMZN":  {"cik": "0001018724", "name": "Amazon.com Inc.",          "sector": "Technology"},
    "NVDA":  {"cik": "0001045810", "name": "NVIDIA Corp.",             "sector": "Technology"},
    "META":  {"cik": "0001326801", "name": "Meta Platforms Inc.",      "sector": "Technology"},
    "INTC":  {"cik": "0000050863", "name": "Intel Corp.",              "sector": "Technology"},
    "ORCL":  {"cik": "0001341439", "name": "Oracle Corp.",             "sector": "Technology"},
    "IBM":   {"cik": "0000051143", "name": "IBM Corp.",                "sector": "Technology"},
    "CSCO":  {"cik": "0000858877", "name": "Cisco Systems Inc.",       "sector": "Technology"},
    "ADBE":  {"cik": "0000796343", "name": "Adobe Inc.",               "sector": "Technology"},
    "CRM":   {"cik": "0001108524", "name": "Salesforce Inc.",          "sector": "Technology"},
    "QCOM":  {"cik": "0000804328", "name": "Qualcomm Inc.",            "sector": "Technology"},
    "TXN":   {"cik": "0000097476", "name": "Texas Instruments Inc.",   "sector": "Technology"},
    "AMD":   {"cik": "0000002488", "name": "Advanced Micro Devices",   "sector": "Technology"},
    "JPM":   {"cik": "0000019617", "name": "JPMorgan Chase & Co.",     "sector": "Financials"},
    "BAC":   {"cik": "0000070858", "name": "Bank of America Corp.",    "sector": "Financials"},
    "WFC":   {"cik": "0000072971", "name": "Wells Fargo & Co.",        "sector": "Financials"},
    "GS":    {"cik": "0000886982", "name": "Goldman Sachs Group Inc.", "sector": "Financials"},
    "MS":    {"cik": "0000895421", "name": "Morgan Stanley",           "sector": "Financials"},
    "BLK":   {"cik": "0001364742", "name": "BlackRock Inc.",           "sector": "Financials"},
    "C":     {"cik": "0000831001", "name": "Citigroup Inc.",           "sector": "Financials"},
    "AXP":   {"cik": "0000004962", "name": "American Express Co.",     "sector": "Financials"},
    "BRK-B": {"cik": "0001067983", "name": "Berkshire Hathaway Inc.",  "sector": "Financials"},
    "USB":   {"cik": "0000036104", "name": "U.S. Bancorp",             "sector": "Financials"},
    "JNJ":   {"cik": "0000200406", "name": "Johnson & Johnson",        "sector": "Healthcare"},
    "UNH":   {"cik": "0000731766", "name": "UnitedHealth Group Inc.",  "sector": "Healthcare"},
    "PFE":   {"cik": "0000078003", "name": "Pfizer Inc.",              "sector": "Healthcare"},
    "ABBV":  {"cik": "0001551152", "name": "AbbVie Inc.",              "sector": "Healthcare"},
    "MRK":   {"cik": "0000310158", "name": "Merck & Co. Inc.",         "sector": "Healthcare"},
    "TMO":   {"cik": "0000097745", "name": "Thermo Fisher Scientific", "sector": "Healthcare"},
    "ABT":   {"cik": "0000001800", "name": "Abbott Laboratories",      "sector": "Healthcare"},
    "LLY":   {"cik": "0000059478", "name": "Eli Lilly & Co.",          "sector": "Healthcare"},
    "BMY":   {"cik": "0000014272", "name": "Bristol-Myers Squibb",     "sector": "Healthcare"},
    "AMGN":  {"cik": "0000318154", "name": "Amgen Inc.",               "sector": "Healthcare"},
    "TSLA":  {"cik": "0001318605", "name": "Tesla Inc.",               "sector": "Consumer Discretionary"},
    "HD":    {"cik": "0000354950", "name": "Home Depot Inc.",          "sector": "Consumer Discretionary"},
    "MCD":   {"cik": "0000063908", "name": "McDonald's Corp.",         "sector": "Consumer Discretionary"},
    "NKE":   {"cik": "0000320187", "name": "Nike Inc.",                "sector": "Consumer Discretionary"},
    "SBUX":  {"cik": "0000829224", "name": "Starbucks Corp.",          "sector": "Consumer Discretionary"},
    "TGT":   {"cik": "0000027419", "name": "Target Corp.",             "sector": "Consumer Discretionary"},
    "LOW":   {"cik": "0000060667", "name": "Lowe's Companies Inc.",    "sector": "Consumer Discretionary"},
    "GM":    {"cik": "0001467858", "name": "General Motors Co.",       "sector": "Consumer Discretionary"},
    "F":     {"cik": "0000037996", "name": "Ford Motor Co.",           "sector": "Consumer Discretionary"},
    "COST":  {"cik": "0000909832", "name": "Costco Wholesale Corp.",   "sector": "Consumer Discretionary"},
    "BA":    {"cik": "0000012927", "name": "Boeing Co.",               "sector": "Industrials"},
    "CAT":   {"cik": "0000018230", "name": "Caterpillar Inc.",         "sector": "Industrials"},
    "GE":    {"cik": "0000040987", "name": "General Electric Co.",     "sector": "Industrials"},
    "MMM":   {"cik": "0000066740", "name": "3M Co.",                   "sector": "Industrials"},
    "HON":   {"cik": "0000773840", "name": "Honeywell Intl. Inc.",     "sector": "Industrials"},
    "UPS":   {"cik": "0001090727", "name": "United Parcel Service",    "sector": "Industrials"},
    "RTX":   {"cik": "0000101829", "name": "Raytheon Technologies",    "sector": "Industrials"},
    "PG":    {"cik": "0000080424", "name": "Procter & Gamble Co.",     "sector": "Consumer Staples"},
    "KO":    {"cik": "0000021344", "name": "Coca-Cola Co.",            "sector": "Consumer Staples"},
    "PEP":   {"cik": "0000077476", "name": "PepsiCo Inc.",             "sector": "Consumer Staples"},
    "WMT":   {"cik": "0000104169", "name": "Walmart Inc.",             "sector": "Consumer Staples"},
    "PM":    {"cik": "0001413329", "name": "Philip Morris Intl.",      "sector": "Consumer Staples"},
    "MO":    {"cik": "0000764180", "name": "Altria Group Inc.",        "sector": "Consumer Staples"},
    "XOM":   {"cik": "0000034088", "name": "Exxon Mobil Corp.",        "sector": "Energy"},
    "CVX":   {"cik": "0000093410", "name": "Chevron Corp.",            "sector": "Energy"},
    "COP":   {"cik": "0001163165", "name": "ConocoPhillips",           "sector": "Energy"},
    "SLB":   {"cik": "0000087347", "name": "Schlumberger Ltd.",        "sector": "Energy"},
    "EOG":   {"cik": "0000821189", "name": "EOG Resources Inc.",       "sector": "Energy"},
    "VZ":    {"cik": "0000732712", "name": "Verizon Communications",   "sector": "Communication"},
    "T":     {"cik": "0000732717", "name": "AT&T Inc.",                "sector": "Communication"},
    "DIS":   {"cik": "0001001039", "name": "Walt Disney Co.",          "sector": "Communication"},
    "NFLX":  {"cik": "0001065280", "name": "Netflix Inc.",             "sector": "Communication"},
    "CMCSA": {"cik": "0001166691", "name": "Comcast Corp.",            "sector": "Communication"},
    "NEE":   {"cik": "0000753308", "name": "NextEra Energy Inc.",      "sector": "Utilities"},
    "DUK":   {"cik": "0001326160", "name": "Duke Energy Corp.",        "sector": "Utilities"},
    "SO":    {"cik": "0000092122", "name": "Southern Co.",             "sector": "Utilities"},
    "AMT":   {"cik": "0001053507", "name": "American Tower Corp.",     "sector": "Real Estate"},
    "PLD":   {"cik": "0001045609", "name": "Prologis Inc.",            "sector": "Real Estate"},
    "LIN":   {"cik": "0001707925", "name": "Linde plc",               "sector": "Materials"},
    "APD":   {"cik": "0000002969", "name": "Air Products & Chemicals", "sector": "Materials"},
    "NEM":   {"cik": "0001164180", "name": "Newmont Corp.",            "sector": "Materials"},
}
print(f"Company universe: {len(SP500_COMPANIES)} companies across 11 sectors")

In [ ]:
def get_10k_filings(cik, ticker):
    """
    Query SEC EDGAR submissions API for a company's 10-K filing history.
    Filters for form type 10-K only (excludes 10-K/A amendments).
    Returns list of dicts with ticker, year, filing_date, accession, index_url.
    """
    url = f"https://data.sec.gov/submissions/CIK{cik.zfill(10)}.json"
    cik_clean = cik.lstrip("0")
    try:
        r = requests.get(url, headers=EDGAR_HEADERS, timeout=15)
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        print(f"  Failed {ticker}: {e}")
        return []

    filings = []
    recent  = data.get("filings", {}).get("recent", {})
    for form, date, acc in zip(
        recent.get("form", []),
        recent.get("filingDate", []),
        recent.get("accessionNumber", [])
    ):
        if form != "10-K":
            continue
        year = int(date[:4])
        if not (START_YEAR <= year <= END_YEAR):
            continue
        acc_clean = acc.replace("-", "")
        index_url = (
            f"https://www.sec.gov/Archives/edgar/data/"
            f"{cik_clean}/{acc_clean}/{acc}-index.htm"
        )
        filings.append({
            "ticker": ticker, "cik": cik,
            "year": year, "filing_date": date,
            "accession": acc, "index_url": index_url,
        })
    return filings

In [ ]:
# Retrieve filing metadata for all companies
# Cached after first run - runtime approximately 5-10 minutes

filing_index_path = DATA_DIR / "filing_index.csv"

if filing_index_path.exists():
    print("Filing index already exists - loading from cache.")
    df_index = pd.read_csv(filing_index_path)
else:
    print(f"Building filing index for {len(SP500_COMPANIES)} companies...")
    all_filings = []
    for ticker, info in tqdm(SP500_COMPANIES.items(), desc="Fetching metadata"):
        filings = get_10k_filings(info["cik"], ticker)
        for f in filings:
            f["name"]   = info["name"]
            f["sector"] = info["sector"]
        all_filings.extend(filings)
        time.sleep(0.12)
    df_index = pd.DataFrame(all_filings)
    df_index.to_csv(filing_index_path, index=False)

print(f"Filing index: {len(df_index):,} filings, {df_index['ticker'].nunique()} companies")
print(f"Year range: {df_index['year'].min()}-{df_index['year'].max()}")

# Phase 2: Extract MD&A Text from Each Filing

Three-step process for each filing:
1. Resolve the primary document URL from the filing index page (strips iXBRL viewer prefix for modern filings)

2. Download and convert the HTML document to plain text

3. Extract the MD&A section using Item 7 heading regex with keyword density fallback

The iXBRL prefix (/ix?doc=) is the most common failure mode for post-2017 filings.
Fetching that URL returns 75 characters of JavaScript redirect rather than the document.

In [ ]:
def fix_ixbrl_url(url):
    """Strip the iXBRL viewer prefix introduced in modern EDGAR filings."""
    if url and "/ix?doc=" in url:
        url = "https://www.sec.gov" + url.split("/ix?doc=")[-1]
    return url

def get_primary_doc_url(index_url, cik):
    """
    Find the primary 10-K document URL from the filing index page.
    Four fallback strategies handle EDGAR formatting variation across years.
    """
    try:
        r = requests.get(index_url, headers=EDGAR_HEADERS, timeout=15)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        cik_clean = str(int(cik))

        all_links = []
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if "/ix?doc=" in href:
                href = href.split("/ix?doc=")[-1]
            if any(href.lower().endswith(ext) for ext in [".htm", ".html", ".txt"]):
                full = href if href.startswith("http") else f"https://www.sec.gov{href}"
                all_links.append((a.get_text(strip=True).lower(), full, href.lower()))

        # Strategy 1: ticker-pattern filename, no exhibit keywords
        for text, full, href in all_links:
            fname = href.split("/")[-1].lower()
            if not any(x in fname for x in ["ex", "exhibit"]) and                any(fname.endswith(ext) for ext in [".htm", ".html"]) and                cik_clean in full:
                return full

        # Strategy 2: explicit 10-K type column in filing table
        for row in soup.find_all("tr"):
            cells = row.find_all("td")
            if len(cells) >= 3:
                doc_type = cells[1].get_text(strip=True).upper().strip()
                if doc_type in ["10-K", "10-K405", "10-KSB"]:
                    link = cells[2].find("a")
                    if link and link.get("href"):
                        href = link["href"]
                        if "/ix?doc=" in href:
                            href = href.split("/ix?doc=")[-1]
                        if not href.startswith("http"):
                            href = "https://www.sec.gov" + href
                        return href

        # Strategy 3: any HTM with CIK, not an exhibit
        candidates = [
            (t, f) for t, f, h in all_links
            if cik_clean in f and not any(
                x in f.lower() for x in
                ["ex-", "exhibit", "ex99", "ex21", "ex23", "ex31", "ex32", "ex97"]
            )
        ]
        if candidates:
            return candidates[0][1]

        # Strategy 4: first non-exhibit HTM link
        for text, full, href_lower in all_links:
            if not any(x in href_lower for x in
                       ["ex-", "exhibit", "ex99", "ex21", "ex23", "ex31", "ex32", "ex97"]):
                return full
    except Exception:
        pass
    return None

In [ ]:
def extract_10k_text(doc_url):
    """Download 10-K filing and convert to plain text."""
    try:
        r = requests.get(doc_url, headers=EDGAR_HEADERS, timeout=30)
        r.raise_for_status()
        if "text/plain" in r.headers.get("content-type", "").lower() or            doc_url.endswith(".txt"):
            return r.text
        soup = BeautifulSoup(r.content, "html.parser")
        for tag in soup(["script", "style", "head", "header", "footer", "nav"]):
            tag.decompose()
        for table in soup.find_all("table"):
            table.replace_with(table.get_text(separator=" "))
        return soup.get_text(separator=" ")
    except Exception:
        return None

def extract_mda(full_text):
    """
    Extract MD&A section using Item 7 regex with keyword density fallback.
    Strategy 1: locate Item 7 heading and extract to Item 7A or Item 8 end marker.
    Strategy 2: find passage with highest financial keyword density.
    Returns None if extracted text is shorter than 300 words.
    """
    if not full_text:
        return None
    text = re.sub(r"\s+", " ", full_text)
    text = re.sub(r"&#\d+;", " ", text)
    text = re.sub(r"&[a-z]+;", " ", text)

    start_patterns = [
        r"(?:item|ITEM)\s*7[\.[\s\-]]+\s*(?:management[\s']+s?\s+discussion|management\s+discussion)",
        r"management[\s']+s?\s+discussion\s+and\s+analysis\s+of\s+financial",
        r"management[\s']+s?\s+discussion\s+and\s+analysis\s+of\s+results",
        r"management[\s']+s?\s+discussion\s+&\s+analysis",
    ]
    end_patterns = [
        r"(?:item|ITEM)\s*7[Aa][\.[\s\-]]+",
        r"(?:item|ITEM)\s*8[\.[\s\-]]+",
        r"quantitative\s+and\s+qualitative\s+disclosures\s+about\s+market",
        r"financial\s+statements\s+and\s+supplementary",
    ]

    start_match = None
    for pat in start_patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            start_match = m
            break

    if start_match:
        search_text = text[start_match.end():]
        end_match = None
        for pat in end_patterns:
            m = re.search(pat, search_text, re.IGNORECASE)
            if m:
                end_match = m
                break
        mda = search_text[:end_match.start()] if end_match else search_text[:20000]
        if len(mda.split()) >= 300:
            return mda.strip()

    financial_keywords = [
        "revenue", "earnings", "operating", "liquidity", "capital",
        "cash flow", "margin", "results of operations", "fiscal year",
        "net income", "gross profit", "selling", "general and administrative",
    ]
    words = text.split()
    best_score, best_start = 0, 0
    for i in range(0, len(words) - 500, 100):
        chunk = " ".join(words[i:i+500]).lower()
        score = sum(1 for kw in financial_keywords if kw in chunk)
        if score > best_score:
            best_score = score
            best_start = i
    if best_score >= 4:
        mda = " ".join(words[max(0, best_start-100):min(len(words), best_start+3000)])
        if len(mda.split()) >= 300:
            return mda.strip()
    return None

def clean_mda(text):
    """Remove non-ASCII artefacts, standalone numbers, and excess whitespace."""
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    text = re.sub(r"\b\d+\b", " ", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()

print("All extraction functions defined")

In [ ]:
# Extract MD&A sections from all filings
# Cached after first run - runtime approximately 30-60 minutes

corpus_path = DATA_DIR / "mda_corpus.csv"

if corpus_path.exists():
    print("MD&A corpus already exists - loading from cache.")
    df_corpus = pd.read_csv(corpus_path)
else:
    print(f"Extracting MD&A from {len(df_index):,} filings...")
    print("Estimated time: 30-60 minutes")
    print()
    records = []
    failed  = 0
    for _, row in tqdm(df_index.iterrows(), total=len(df_index), desc="Extracting"):
        doc_url   = get_primary_doc_url(row["index_url"], row["cik"])
        if not doc_url: failed += 1; continue
        full_text = extract_10k_text(doc_url)
        if not full_text: failed += 1; continue
        mda = extract_mda(full_text)
        if not mda: failed += 1; continue
        mda_clean = clean_mda(mda)
        records.append({
            "ticker":      row["ticker"],
            "name":        row["name"],
            "sector":      row["sector"],
            "year":        row["year"],
            "filing_date": row["filing_date"],
            "text":        mda_clean,
            "word_count":  len(mda_clean.split()),
        })
        time.sleep(0.12)
    df_corpus = pd.DataFrame(records)
    df_corpus.to_csv(corpus_path, index=False)
    print(f"Corpus saved: {len(df_corpus):,} successful, {failed:,} failed")

print(f"Corpus: {len(df_corpus):,} documents | {df_corpus['ticker'].nunique()} companies")
print(f"Year range: {df_corpus['year'].min()}-{df_corpus['year'].max()}")
print(df_corpus[["ticker", "year", "sector", "word_count"]].head(10).to_string(index=False))

# Done

The data/ folder now contains:
- filing_index.csv: all 10-K filing URLs retrieved from EDGAR
- mda_corpus.csv: 417 extracted and cleaned MD&A documents

You can now run the main analysis notebook (01_SP500_Sentiment_Analysis.ipynb) which loads these files directly.

### Key Design Decisions

iXBRL Resolution: Post-2017 EDGAR filings use the URL prefix /ix?doc= which returns a JavaScript viewer page rather than the document. Stripping this prefix was the single most impactful fix, raising the extraction success rate from approximately 5% to over 99%.

Dual Extraction Strategy: The primary regex approach matches Item 7 heading variants. The keyword density fallback handles cases where headings are embedded in formatting that survives HTML stripping. Together they achieve a 417/418 success rate (99.8%).

300-Word Minimum: Documents below 300 words are discarded as likely extraction failures. This threshold was chosen to exclude clearly partial extractions while retaining shorter but valid MD&A sections from smaller or earlier filings.